# Training & Evaluation - Classifying Spam Emails

Notebook nay tong hop phan train/evaluate model spam/not spam cho project. Neu file train lon chua duoc tai qua Git LFS, notebook se dung metric, prediction va figure da co trong `reports/` thay vi co retrain roi loi.

## 1. Kết quả lọc dữ liệu hiện tại

- Input rows: `296196`
- Clean rows: `294718`
- Removed rows: `1478`
- Label trước lọc: `{'0': 148098, '1': 148098}`
- Label sau lọc/cân bằng: `{'0': 147359, '1': 147359}`
- Lỗi chính: `{'short_text': 1156}`

In [ ]:
from pathlib import Path
import json
import sys
import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

DATA_PATH = PROJECT_ROOT / "data" / "processed" / "combined_balanced_clean.csv"
REPORTS_DIR = PROJECT_ROOT / "reports"
FIGURES_DIR = REPORTS_DIR / "figures"


def read_json(path):
    path = Path(path)
    return json.loads(path.read_text(encoding="utf-8")) if path.exists() else {}


def is_real_csv(path: Path) -> bool:
    if not path.exists() or path.stat().st_size < 1024:
        return False
    first_line = path.open("r", encoding="utf-8", errors="replace").readline().strip().lower()
    return not first_line.startswith("version https://git-lfs.github.com/spec")

quality_report = read_json(REPORTS_DIR / "data_quality_report.json")
train_report = read_json(REPORTS_DIR / "model_train_report.json")
DATA_AVAILABLE = is_real_csv(DATA_PATH)

print("Project root:", PROJECT_ROOT)
print("Data path:", DATA_PATH)
print("Real training CSV available:", DATA_AVAILABLE)
if not DATA_AVAILABLE:
    print("Large training data is missing or is a Git LFS pointer; using existing reports/metrics.")

In [ ]:
if DATA_AVAILABLE:
    dataset = pd.read_csv(DATA_PATH)
    print(dataset.shape)
    display(dataset.head())
else:
    dataset = pd.DataFrame()
    summary = {
        "input_rows": quality_report.get("input_rows"),
        "clean_rows": quality_report.get("clean_rows"),
        "removed_rows": quality_report.get("removed_rows"),
        "train_rows_in_report": train_report.get("train_rows"),
        "test_rows_in_report": train_report.get("test_rows"),
    }
    display(pd.DataFrame([summary]))

In [ ]:
if not dataset.empty and "label" in dataset.columns:
    label_counts = dataset["label"].value_counts().sort_index().rename(index={0: "not spam", 1: "spam"})
else:
    clean_counts = quality_report.get("clean_label_counts", {})
    label_counts = pd.Series({
        "not spam": clean_counts.get("0", clean_counts.get(0)),
        "spam": clean_counts.get("1", clean_counts.get(1)),
    }).dropna().astype(int)

label_percent = (label_counts / label_counts.sum() * 100).round(2)
pd.DataFrame({"count": label_counts, "percent": label_percent})

## 2. Kiểm tra naming/source code

- Số file Python đã kiểm tra: `11`
- Số lỗi đặt tên: `0`
- Quy tắc: file/function/variable dùng `snake_case`, constant dùng `UPPER_CASE`, class dùng `PascalCase`.

In [ ]:
style_report_path = REPORTS_DIR / "code_style_name_check.json"
style_report = json.loads(style_report_path.read_text(encoding="utf-8"))
style_report["naming_issues"]

## 3. Train model

Ba model được train cùng pipeline `TfidfVectorizer + classifier`:

- `naive_bayes`: nhanh, baseline tốt cho text classification.
- `logistic_regression`: cân bằng giữa tốc độ, độ chính xác và khả năng giải thích.
- `linear_svm`: thường mạnh với TF-IDF và dữ liệu văn bản lớn.

Metric ưu tiên là `recall_spam` và `f1_spam` vì bỏ sót spam nguy hiểm hơn chặn nhầm một số email tốt.

In [ ]:
metrics_path = REPORTS_DIR / "model_metrics.csv"

if DATA_AVAILABLE:
    from src.model_train import run_modeling_pipeline

    SAMPLE_SIZE = 60000
    metrics_table = run_modeling_pipeline(sample_size=SAMPLE_SIZE)
else:
    print("Skipping retrain because the real training CSV is unavailable; reading reports/model_metrics.csv.")
    metrics_table = pd.read_csv(metrics_path)

metrics_table.sort_values(["f1_spam", "recall_spam"], ascending=False)

## 4. Metric sau train gần nhất

| model               | accuracy | precision_spam | recall_spam | f1_spam  |
| ------------------- | -------- | -------------- | ----------- | -------- |
| linear_svm          | 0.979167 | 0.979007       | 0.979333    | 0.97917  |
| logistic_regression | 0.974    | 0.970394       | 0.977833    | 0.974099 |
| naive_bayes         | 0.963417 | 0.977339       | 0.948833    | 0.962875 |

In [ ]:
metrics_path = REPORTS_DIR / "model_metrics.csv"
metrics = pd.read_csv(metrics_path)
metrics.sort_values(["f1_spam", "recall_spam"], ascending=False)

In [ ]:
from src.model_evaluate import evaluate_from_predictions_csv

comparison = evaluate_from_predictions_csv(
    predictions_csv=REPORTS_DIR / "model_predictions.csv",
    true_col="label",
    pred_cols=["naive_bayes", "logistic_regression", "linear_svm"],
    output_dir=FIGURES_DIR,
)
comparison

## 4.1 Permutation importance cho manual features

Phan nay bo sung yeu cau giai thich do quan trong feature. Vi model cuoi trong `src/model_train.py` la pipeline TF-IDF sparse, notebook dung mot mo hinh Logistic Regression nho tren nhom manual features de do **permutation importance**: xao tron tung feature tren tap test va do muc giam F1. Ket qua nay dung de giai thich tin hieu thu cong, khong phai de thay the model Linear SVM cuoi.

In [ ]:
from IPython.display import Image, display

importance_csv = FIGURES_DIR / "manual_feature_permutation_importance.csv"
importance_png = FIGURES_DIR / "manual_feature_permutation_importance.png"

if importance_csv.exists():
    importance_df = pd.read_csv(importance_csv)
    display(importance_df)
else:
    print("Permutation importance CSV not found. Run 02_eda.ipynb or regenerate figures first.")

if importance_png.exists():
    display(Image(filename=str(importance_png)))
else:
    print("Permutation importance image not found:", importance_png)

## 5. Learning curve

Ba biểu đồ learning curve dưới đây cho thấy F1-score thay đổi khi tăng số lượng mẫu train:

- Đường đỏ: F1-score trên tập train.
- Đường xanh: F1-score cross-validation.
- Khoảng cách giữa hai đường càng nhỏ thì model càng ít overfitting.

Trong các biểu đồ này, `linear_svm` và `logistic_regression` có đường cross-validation tăng ổn định khi thêm dữ liệu. `naive_bayes` dao động nhiều hơn và F1 thấp hơn, nên phù hợp làm baseline hơn là model cuối.

In [ ]:
from IPython.display import Image, display

learning_curve_images = {
    "Linear SVM": FIGURES_DIR / "learning_curve_linear_svm.png",
    "Logistic Regression": FIGURES_DIR / "learning_curve_logistic_regression.png",
    "Naive Bayes": FIGURES_DIR / "learning_curve_naive_bayes.png",
}

for title, image_path in learning_curve_images.items():
    print(title, image_path)
    if image_path.exists():
        display(Image(filename=str(image_path)))
    else:
        print("Chưa tìm thấy ảnh:", image_path)

### 5.1 Ảnh learning curve trong báo cáo

![Learning Curve - Linear SVM](../reports/figures/learning_curve_linear_svm.png)

![Learning Curve - Logistic Regression](../reports/figures/learning_curve_logistic_regression.png)

![Learning Curve - Naive Bayes](../reports/figures/learning_curve_naive_bayes.png)

## 6. Confusion matrix

Các ảnh dưới đây được sinh tự động từ `src/model_evaluate.py`. Nếu nhóm muốn dùng ảnh ngoài cho báo cáo cuối, có thể thay các file trong `reports/figures/`.

In [ ]:
from IPython.display import Image, display

for name in ["naive_bayes", "logistic_regression", "linear_svm"]:
    image_path = FIGURES_DIR / f"{name}_confusion_matrix.png"
    print(name, image_path)
    if image_path.exists():
        display(Image(filename=str(image_path)))

## 7. Demo predict

File `src/predict.py` load model tốt nhất từ `models/spam_classifier.joblib` và trả về nhãn `spam` hoặc `not spam`, kèm `spam_score` trong khoảng 0-1.

In [ ]:
from src.predict import predict_email

sample_emails = [
    "Congratulations! You won a free iPhone. Click now to claim your prize.",
    "Hi team, please confirm your availability for the meeting tomorrow.",
    "Urgent account warning: verify your password immediately or your mailbox will be suspended.",
]

[predict_email(email) for email in sample_emails]

## 8. Ket luan nhanh

- Theo report hien co, dataset clean da can bang 2 class va da loai dong qua ngan/loi.
- `linear_svm` dang la model tot nhat trong lan train mau 60,000 dong.
- Khi co du Git LFS objects, co the bat retrain bang cach chay lai notebook voi CSV train that trong `data/processed/`.